In [2]:
import warnings
warnings.filterwarnings('ignore')
import polars as pl
import polars.selectors as cs
import numpy as np
import math
import seaborn as sns
import os 
import re
import matplotlib.pyplot as plt
import gzip
import matplotlib.colors as mcolors
from scipy import stats
pl.Config.set_fmt_str_lengths(50)
pl.Config().set_tbl_rows(50)
sns.set_style(style='white')
warnings.filterwarnings('ignore')

1. Validate that the luciferase assay recapitulates ccMPRA-identified activity and provides baseline measurements for each promoter.
-- 20 strong enhancers (10 encode and 10 non-encode annotated) + the promoters alone
-- 20 strong silencers + the promoters alone
= 80 sequences
2. Test the promoter-dependent activity.
-- 3 CREs that act as enhancers or silencers based on the promoter + the promoters alone
-- 3 CREs that act as enhancer for 1 promoter and no effect on the other promoter + the promoters alone
-- 3 CREs that act as silencer for 1 promoter and no effect on the other promoter + the promoters alone
= 36 sequences
3. Then the reviewer 2 was worried about the effect of the length variation, coordinates variation and distance from H3k27ac peaks so I was thinking:
-- Reuse 5 strong enhancers and 5 strong silencers that we tested in point 1 and test 2 additional lengths for them (20). No need to redo the promoter alone.
-- Among those 10 select 5 and test the effect of 2 shifts of the position as compared to the H3k27ac peak (10). No need to redo the promoter alone.
-- Among the promoters we used in point 1, we select 5 and we try an other length for them, either alone either with their CRE (fixe length, same then point 1) (10).
= 40 sequences

## Filtering

Filtering on:
- both parts at least 100 bp
- CRE is not an encode PLS
- promoter contains TSS


In [3]:
cd = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/"
data = pl.read_csv(os.path.join(cd,"results/MPRA_analysis/CMPRA5/labeled_data_promoteroa_OA.tsv"), separator="\t")
data = data.filter(pl.col("right_bin").str.contains("null").not_()).rename({"OE": "CRE", "nr_reads": "nr_barcodes", "dist": "distance", "interaction": "ENCODE_labels"})
data = data.filter(pl.col("label") != "other - other")
data = data.filter(pl.col("any_tss") == "yes")
data = data.with_columns(
	CRE_length = pl.col("CRE").str.split("-").list.get(2).cast(pl.Int64) - pl.col("CRE").str.split("-").list.get(1).cast(pl.Int64),
	promoter_length = pl.col("promoter").str.split("-").list.get(2).cast(pl.Int64) - pl.col("promoter").str.split("-").list.get(1).cast(pl.Int64)
)
data = data.filter((pl.col("CRE_length") >= 100) & (pl.col("promoter_length") >= 100))
data = data.with_columns(pl.when(pl.all_horizontal(pl.any_horizontal(cs.matches("screen").str.contains("PLS")) & pl.any_horizontal(cs.matches("screen").is_null())))
			 			.then(pl.lit("PLS - undefined"))
						.when(pl.all_horizontal(pl.any_horizontal(cs.matches("screen").str.contains("ELS")) & pl.any_horizontal(cs.matches("screen").is_null())))
						.then(pl.lit("ELS - undefined"))
						 .when(pl.all_horizontal(cs.matches("screen").is_null())).then(pl.lit("undefined")).otherwise(pl.col("ENCODE_labels"))
						 .alias("ENCODE_labels"))
data = data.filter(pl.col("ENCODE_labels").str.contains("PLS - PLS").not_()).select(~cs.matches("left|right|tss|Val|std"))

## Enhancers and silencers

In [4]:
silencers = pl.concat([data.filter(pl.col("ENCODE_labels") == "PLS - undefined").sort("z_score").head(200),
						data.filter(pl.col("ENCODE_labels") == "PLS - ELS").sort("z_score").head(200)])
enhancers = pl.concat([data.filter(pl.col("ENCODE_labels") == "PLS - undefined").sort("z_score", descending=True).head(200),
						data.filter(pl.col("ENCODE_labels") == "PLS - ELS").sort("z_score", descending=True).head(200)])

## Multi interacting CREs

In [5]:
multi_prom = data.filter(pl.col("promoter").n_unique().over("CRE") >= 2) \
	.filter((pl.col("z_score").max().over("CRE") > 1.5) | (pl.col("z_score").min().over("CRE") < -1.5)) \
	.with_columns(activity_difference = np.abs(pl.col("z_score").max().over("CRE") - pl.col("z_score").min().over("CRE")))\
		.sort("activity_difference", descending=True)

multi_prom = multi_prom.filter(pl.col("CRE").is_in(multi_prom.select("CRE").unique(maintain_order=True).head(100)["CRE"]))

## Getting bin order of original sequences for enhancers and silencers

In [7]:
!zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/mprasnakeflow/results/assigned_bcs_part1and2.tsv.gz' | cut -f -2,4,6,8,10,12,14 | awk 'NR > 1 && NF >=5' | cut -f 2 > temp.seqids.tsv

In [8]:
pl.concat([silencers, enhancers]).select(pl.col("CRE")).write_csv("CREs.temp.ids.tsv", include_header=False)
pl.concat([silencers, enhancers]).select(pl.col("promoter")).write_csv("promoter.temp.ids.tsv", include_header=False)

In [9]:
seq_ids = "temp.seqids.tsv"
cre_ids = "CREs.temp.ids.tsv"
prom_ids = "promoter.temp.ids.tsv"

In [10]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf CREs.temp.ids.tsv | grep -Fwf promoter.temp.ids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.actualbins.tsv

In [11]:
# Check uniqueness
! echo $(cut -f -2 temp.actualbins.tsv | sort | uniq |  wc -l) $(wc -l < temp.actualbins.tsv)
! echo $(wc -l < CREs.temp.ids.tsv) # looks like there are some incorrect bins

1205 1205
800


In [6]:
actual_bins = pl.read_csv("temp.actualbins.tsv", separator="\t", has_header=False).rename({"column_1": "left_bin", "column_2": "right_bin"}) 

In [7]:
actual_bins.with_columns(column_3 = pl.col("column_3").str.split(",").list.len()).head()

left_bin,right_bin,column_3
str,str,u32
"""chr1-11805849-11806206-+""","""chr1-11842196-11842568-.""",12
"""chr1-12392815-12393089-.""","""chr1-12618206-12618597--""",14
"""chr1-12393090-12393210-.""","""chr1-12618206-12618597--""",13
"""chr1-12618206-12618597--""","""chr1-12350334-12350578-.""",17
"""chr1-12618206-12618597--""","""chr1-12565474-12565661-.""",33


In [132]:
! grep "chr1-44988261-44988729-." temp.actualbins.tsv

chr1-44988261-44988729-.	chr1-45012148-45012412-+	m84066_240516_024507_s2/116200304/ccs,m84066_240516_024507_s2/117968625/ccs,m84066_240516_024507_s2/119738712/ccs,m84066_240516_024507_s2/123341587/ccs,m84066_240516_024507_s2/125175647/ccs,m84066_240516_024507_s2/134353403/ccs,m84066_240516_024507_s2/161482149/ccs,m84066_240516_024507_s2/197003724/ccs,m84066_240516_024507_s2/236719141/ccs,m84066_240516_024507_s2/264769070/ccs,m84066_240516_024507_s2/53546331/ccs,m84066_240516_024507_s2/67899501/ccs,m84066_240516_024507_s2/69010563/ccs,m84066_240623_102831_s3/101520839/ccs,m84066_240623_102831_s3/146478915/ccs,m84066_240623_102831_s3/157025067/ccs,m84066_240623_102831_s3/169610701/ccs,m84066_240623_102831_s3/239473830/ccs,m84066_240623_102831_s3/83168331/ccs,m84066_240623_122801_s1/17567107/ccs,m84066_240623_122801_s1/247466350/ccs,m84066_240623_122801_s1/37097751/ccs,m84066_240623_122801_s1/37687676/ccs
chr1-45012148-45012412-+	chr1-44988261-44988729-.	m84066_240516_024507_s2/34406993/

In [ ]:
# --------- Code if we want to try multiple bin orders -----------

# Get the silencers with the bin order in which they were sequenced, and additionally in a CRE - promoter order
silencers_with_bin_order = silencers.select(pl.col("CRE").alias("left_bin"), pl.col("promoter").alias("right_bin"), pl.all())
silencers_with_bin_order = pl.concat([silencers_with_bin_order, 
									  actual_bins.select(pl.exclude("column_3")).join(silencers, left_on=["left_bin", "right_bin"], right_on=["promoter", "CRE"], coalesce = False)])

enhancers_with_bin_order = enhancers.select(pl.col("CRE").alias("left_bin"), pl.col("promoter").alias("right_bin"), pl.all())
enhancers_with_bin_order = pl.concat([enhancers_with_bin_order, 
									  actual_bins.select(pl.exclude("column_3")).join(enhancers, left_on=["left_bin", "right_bin"], right_on=["promoter", "CRE"], coalesce = False)])
enhancers_with_bin_order.height


In [9]:
# --------- Code if we want to test CRE - promoter order only -----------

silencers_with_bin_order = actual_bins.select(pl.exclude("column_3")).join(silencers, left_on=["left_bin", "right_bin"], right_on=["CRE", "promoter"], coalesce = False)

enhancers_with_bin_order = actual_bins.select(pl.exclude("column_3")).join(enhancers, left_on=["left_bin", "right_bin"], right_on=["CRE", "promoter"], coalesce = False)
silencers_with_bin_order.height, silencers.height

(257, 400)

In [14]:
silencers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - undefined").height, enhancers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - undefined").height
#silencers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - ELS").height, enhancers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - ELS").height

(125, 113)

In [55]:
final_silencers = pl.concat([silencers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - undefined").sort("z_score").head(10),
						silencers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - ELS").sort("z_score").head(10)])
final_enhancers = pl.concat([enhancers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - undefined").sort("z_score", descending=True).head(10),
						enhancers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - ELS").sort("z_score", descending=True).head(10)])
final_enhancers

## Getting bin order of original sequences for multi-promoter CREs

In [63]:
multi_prom.select(pl.col("CRE")).write_csv("CREs.multiprom.temp.ids.tsv", include_header=False)
multi_prom.select(pl.col("promoter")).write_csv("promoter.multiprom.temp.ids.tsv", include_header=False)
cre_ids = "CREs.multiprom.temp.ids.tsv"
prom_ids = "promoter.multiprom.temp.ids.tsv"

In [64]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf CREs.multiprom.temp.ids.tsv | grep -Fwf promoter.multiprom.temp.ids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.multiprom.actualbins.tsv

In [65]:
actual_bins = pl.read_csv("temp.multiprom.actualbins.tsv", separator="\t", has_header=False).rename({"column_1": "left_bin", "column_2": "right_bin"})

In [34]:
actual_bins.with_columns(column_3 = pl.col("column_3").str.split(",").list.len()).head()

left_bin,right_bin,column_3
str,str,u32
"""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",16
"""chr14-20955217-20955500-.""","""chr14-20890827-20891165-+""",16
"""chr14-35115858-35116161-.""","""chr14-35121748-35122080--""",70
"""chr14-35121748-35122080--""","""chr14-35115858-35116161-.""",67
"""chr14-35122081-35122903--""","""chr14-35115858-35116161-.""",41


Dont forget: there are more sequences in the bin, that are not used in the end, so which are not counted in nr_seqs

In [36]:
multi_prom.filter(pl.col("CRE") == "chr14-20955217-20955500-.")

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
-1.086703,11,2,"""target - other""","""PLS - undefined""",267082.0,"""ANG""","""no effect""","""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",-1.379707,0.916286,283,357,4.60108
0.618555,11,1,"""negative - other""","""PLS - undefined""",64362.5,"""RNASE3""","""upregulating""","""chr14-20890827-20891165-+""","""chr14-20955217-20955500-.""",-1.287388,5.517367,283,338,4.60108


In [80]:
multi_prom_with_binorder = actual_bins.select(pl.exclude("column_3")).join(multi_prom, left_on=["left_bin", "right_bin"], right_on=["CRE", "promoter"], coalesce = False)
multi_prom_with_binorder.sort("activity_difference", descending=True).head()


left_bin,right_bin,logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
str,str,f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
"""chr14-20955217-20955500-.""","""chr14-20890827-20891165-+""",0.618555,11,1,"""negative - other""","""PLS - undefined""",64362.5,"""RNASE3""","""upregulating""","""chr14-20890827-20891165-+""","""chr14-20955217-20955500-.""",-1.287388,5.517367,283,338,4.60108
"""chr20-17963417-17963734-.""","""chr20-17968096-17968866-+""",-1.259062,8,3,"""target - other""","""PLS - ELS""",4905.5,"""MGME1""","""downregulating""","""chr20-17968096-17968866-+""","""chr20-17963417-17963734-.""",-0.157991,-2.240839,317,770,4.326135
"""chr14-35115858-35116161-.""","""chr14-35121748-35122080--""",0.419135,30,6,"""target - other""","""PLS - undefined""",5904.5,"""PPP2R3C""","""no effect""","""chr14-35121748-35122080--""","""chr14-35115858-35116161-.""",0.046627,1.151496,303,332,3.946451
"""chr15-34095485-34095931-.""","""chr15-34101991-34102521--""",-1.83007,5,1,"""target - other""","""PLS - undefined""",6548.0,"""EMC7""","""downregulating""","""chr15-34101991-34102521--""","""chr15-34095485-34095931-.""",0.189146,-3.633559,446,530,3.777777
"""chr2-99128649-99128811-.""","""chr2-99180852-99181315-+""",1.34942,6,2,"""positive - other""","""PLS - undefined""",52353.5,"""MRPL30""","""no effect""","""chr2-99180852-99181315-+""","""chr2-99128649-99128811-.""",1.026821,0.626273,162,463,3.65902


## Orientation of CREs

In [10]:
pl.concat([silencers_with_bin_order, enhancers_with_bin_order]).select(pl.col("CRE")).write_csv("CREs.withbinorder.temp.ids.tsv", include_header=False)
pl.concat([silencers_with_bin_order, enhancers_with_bin_order]).select(pl.col("promoter")).write_csv("promoter.withbinorder.temp.ids.tsv", include_header=False)

In [ ]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" |  grep -Fwf <(paste CREs.withbinorder.temp.ids.tsv promoter.withbinorder.temp.ids.tsv) | \
			> temp.actualbins.withbinorder.tsv

In [ ]:
#### Oke shit het probleem is: dit singlebinssmalleroverlap_OA bestanden werken niet, omdat hier de bin order gesorteerd is ipv de originele bin order.
#### Wacht nee dat klopt niet!!!! Dit is nog voor het sorteren en mergen, dus dit is WEL de originele bin order.

In [91]:
!zcat "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz" \
	| grep -Fwf <(sed 's/>//' temp.seqids.tsv) | awk -v OFS="-" '{print $1,$2,$3"\t"$5"\t"$4}'   | grep -Fwf <(cut -f 1 temp.actualbins.withbinorder.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g' ) > left.temp.tsv

In [92]:
!zcat "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz" \
	| grep -Fwf <(sed 's/>//' temp.seqids.tsv) | awk -v OFS="-" '{print $1,$2,$3"\t"$5"\t"$4}'   | grep -Fwf <(cut -f 2 temp.actualbins.withbinorder.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g' ) > right.temp.tsv

In [122]:
!join -1 3 -2 3 -o 1.1 2.1 1.2 2.2 1.3 <(sort -k3,3 left.temp.tsv) <(sort -k3,3 right.temp.tsv) -t $'\t' | awk -v OFS="\t" '{print $1,$2,$3,$4,$1"-"$3,$2"-"$4,$5}'\
	| grep -v -Fwf <(cut -f -2 temp.actualbins.withbinorder.tsv | sed 's/-\.//g') | cut -f -6 | sort | uniq -c > temp.promoter.orientation.counts.tsv

In [158]:
!awk '{ \
	key = $2 FS $7; \
	count[key] += $1; \
	row_count[key]++; \
	lines[key,row_count[key]] = $0; \
	val6[key,row_count[key]] = $6; \
	val7[key,row_count[key]] = $7; \
	cnt[key,row_count[key]] = $1 \
} \
END { \
	for (k in count) { \
		if (row_count[k] == 1) { \
			print val6[k,1], val7[k,1] \
		} else if (row_count[k] == 2) { \
			if (cnt[k,1] / count[k] > 0.5) { \
				print val6[k,1], val7[k,1] \
			} else if (cnt[k,2] / count[k] > 0.5) { \
				print val6[k,2], val7[k,2] \
			} else { \
				print "ambiguous", "ambiguous" ; \
			} \
		} \
	} \
} \
' temp.promoter.orientation.counts.tsv > temp.finalbins.tsv # final number of promoters with unique orientation

			# } else { \
			# 	print count[k], lines[k,1], "ambiguous", "ambiguous" ; \
			# 	print count[k], lines[k,2], "ambiguous", "ambiguous" ; \
			# }\

In [234]:
final_bins = pl.read_csv("temp.finalbins.tsv", separator=" ", has_header=False)\
.rename({"column_1": "CRE_withorientation", "column_2": "promoter"}).with_columns(
	CRE = pl.col("CRE_withorientation").str.replace("--","-.") \
		.str.replace("-\+","-."))

enhancers_with_orientation = enhancers_with_bin_order.join(final_bins, on=["promoter", "CRE"]) 
silencers_with_orientation = silencers_with_bin_order.join(final_bins, on=["promoter", "CRE"]) \



## Orientation of CREs multi promoter

In [178]:
multi_prom.select(pl.col("CRE")).write_csv("CREs.multiprom.withbinorder.temp.ids.tsv", include_header=False)
multi_prom.select(pl.col("promoter")).write_csv("promoter.multiprom.withbinorder.temp.ids.tsv", include_header=False)

In [179]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/prommoteroa_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf CREs.multiprom.withbinorder.temp.ids.tsv | grep -Fwf promoter.multiprom.withbinorder.temp.ids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.actualbins.multiprom.withbinorder.tsv

In [180]:
!zcat "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz" \
	| grep -Fwf <(sed 's/>//' temp.seqids.tsv) | awk -v OFS="-" '{print $1,$2,$3"\t"$5"\t"$4}'   | grep -Fwf <(cut -f 1 temp.actualbins.multiprom.withbinorder.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g' ) > left.multiprom.temp.tsv

In [181]:
!zcat "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz" \
	| grep -Fwf <(sed 's/>//' temp.seqids.tsv) | awk -v OFS="-" '{print $1,$2,$3"\t"$5"\t"$4}'   | grep -Fwf <(cut -f 2 temp.actualbins.multiprom.withbinorder.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g' ) > right.multiprom.temp.tsv

In [186]:
!join -1 3 -2 3 -o 1.1 2.1 1.2 2.2 1.3 <(sort -k3,3 left.multiprom.temp.tsv) <(sort -k3,3 right.multiprom.temp.tsv) -t $'\t' | awk -v OFS="\t" '{print $1,$2,$3,$4,$1"-"$3,$2"-"$4,$5}'\
	| grep -v -Fwf <(cut -f -2 temp.actualbins.multiprom.withbinorder.tsv | sed 's/-\.//g') | cut -f -6 | sort | uniq -c > temp.multiprom.part1.promoter.orientation.counts.tsv

In [187]:
!join -1 3 -2 3 -o 1.1 2.1 1.2 2.2 1.3 <(sort -k3,3 left.multiprom.temp.tsv) <(sort -k3,3 right.multiprom.temp.tsv) -t $'\t' | awk -v OFS="\t" '{print $1,$2,$3,$4,$1"-"$3,$2"-"$4,$5}'\
	| grep -v -Fwf <(awk '{print $2"\t"$1}' temp.actualbins.multiprom.withbinorder.tsv | sed 's/-\.//g') | cut -f -6 | sort | uniq -c > temp.multiprom.part2.promoter.orientation.counts.tsv

In [188]:
!wc -l temp.multiprom.part1.promoter.orientation.counts.tsv temp.multiprom.part2.promoter.orientation.counts.tsv

  298 temp.multiprom.part1.promoter.orientation.counts.tsv
  384 temp.multiprom.part2.promoter.orientation.counts.tsv
  682 total


In [197]:
# I need to sort the bins vertically again from left to right, because we are doing it bin order agnostic for the multi prom CREs

!cat temp.multiprom.part1.promoter.orientation.counts.tsv temp.multiprom.part2.promoter.orientation.counts.tsv \
	| awk  -v OFS="\t" '{if ($1 > $2) print $1,$3,$2,$5,$4,$7,$6; else print $0}' | sort | uniq -c | awk -v OFS="\t" '{print $1*$2,$3,$4,$5,$6,$7,$8}' \
		> temp.multiprom.promoter.orientation.counts.tsv

In [ ]:
# Always putting the CRE first, so it's easier for the future analysis
!awk -v OFS="\t" 'NR==FNR {proms[$0]++; next} {if ($6 in proms) print $1,$3,$2,$5,$4,$7,$6; else print $0;}' "promoter.multiprom.withbinorder.temp.ids.tsv" \
	temp.multiprom.promoter.orientation.counts.tsv > temp.multiprom.promoter.orientation.counts.crefirst.tsv

In [222]:
!grep "chr14-20688098-20688455-+" temp.multiprom.promoter.orientation.counts.crefirst.tsv

2	chr14-20673227-20673454	chr14-20688098-20688455	-	+	chr14-20673227-20673454--	chr14-20688098-20688455-+
1	chr14-20955217-20955500	chr14-20688098-20688455	+	+	chr14-20955217-20955500-+	chr14-20688098-20688455-+
1	chr14-20955217-20955500	chr14-20688098-20688455	-	+	chr14-20955217-20955500--	chr14-20688098-20688455-+
4	chr14-20683835-20684397	chr14-20688098-20688455	+	+	chr14-20683835-20684397-+	chr14-20688098-20688455-+
6	chr14-20688477-20688991	chr14-20688098-20688455	-	+	chr14-20688477-20688991--	chr14-20688098-20688455-+
8	chr14-20688477-20688991	chr14-20688098-20688455	-	+	chr14-20688477-20688991--	chr14-20688098-20688455-+
12	chr14-20688098-20688455	chr14-20688477-20688991	+	+	chr14-20688098-20688455-+	chr14-20688477-20688991-+
14	chr14-20688477-20688991	chr14-20688098-20688455	+	+	chr14-20688477-20688991-+	chr14-20688098-20688455-+


In [223]:
# Part 1
!awk '{ \
	key = $2 FS $7; \
	count[key] += $1; \
	row_count[key]++; \
	lines[key,row_count[key]] = $0; \
	val2[key,row_count[key]] = $2; \
	val5[key,row_count[key]] = $5; \
	val6[key,row_count[key]] = $6; \
	val7[key,row_count[key]] = $7; \
	cnt[key,row_count[key]] = $1 \
} \
END { \
	for (k in count) { \
		if (row_count[k] == 1) { \
			print val6[k,1], val7[k,1] \
		} else if (row_count[k] == 2) { \
			if (cnt[k,1] / count[k] > 0.5) { \
				print val6[k,1], val7[k,1] \
			} else if (cnt[k,2] / count[k] > 0.5) { \
				print val6[k,2], val7[k,2] \
			} else { \
				print val2[k,1]"-"val5[k,1], val7[k,1] ; \
			} \
		} \
	} \
} \
' temp.multiprom.promoter.orientation.counts.crefirst.tsv  > temp.multiprom.finalbins.tsv # final number of promoters with unique orientation

			# } else { \
			# 	print count[k], lines[k,1], "ambiguous", "ambiguous" ; \
			# 	print count[k], lines[k,2], "ambiguous", "ambiguous" ; \
			# }\

In [228]:
!grep "chr14-20955217-20955500"  temp.multiprom.finalbins.tsv

chr14-20688477-20688991-- chr14-20955217-20955500--
chr14-20955217-20955500-+ chr14-20688098-20688455-+
chr14-20955217-20955500-+ chr14-20688477-20688991-+
chr14-20891166-20891415-- chr14-20955217-20955500--


## Selecting the dual function CREs

In [229]:
dual_function = multi_prom.sort("activity_difference", descending=True)\
.filter((pl.col("z_score").min().over("CRE") < -1.5) & (pl.col("z_score").max().over("CRE") > 1.5)\
& (pl.col("promoter") != "chr17-75261786-75262174--"))
dual_function = dual_function.filter(pl.col("CRE").is_in(dual_function.select("CRE").unique(maintain_order=True).head(3)["CRE"]))
dual_function

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
-1.259062,8,3,"""target - other""","""PLS - ELS""",4905.5,"""MGME1""","""downregulating""","""chr20-17968096-17968866-+""","""chr20-17963417-17963734-.""",-0.157991,-2.240839,317,770,4.326135
0.005819,5,2,"""target - other""","""PLS - ELS""",4905.5,"""SNX5""","""upregulating""","""chr20-17968096-17968866--""","""chr20-17963417-17963734-.""",-0.887512,2.085295,317,770,4.326135
1.227543,5,2,"""target - other""","""PLS - undefined""",8630.0,"""GGA3""","""no effect""","""chr17-75261404-75261785--""","""chr17-75252859-75253070-.""",0.602591,1.722083,211,381,4.307408
-0.200868,11,3,"""target - other""","""PLS - undefined""",9015.5,"""MRPS7""","""no effect""","""chr17-75261786-75262174-+""","""chr17-75252859-75253070-.""",0.440545,-1.836872,211,388,4.307408
1.044954,7,2,"""target - other""","""PLS - ELS""",255839.5,"""TOMM6""","""no effect""","""chr6-41787436-41787975-+""","""chr6-42043482-42043608-.""",0.463298,1.87923,126,539,3.529796
-0.94513,5,2,"""target - other""","""PLS - ELS""",122455.5,"""BYSL""","""no effect""","""chr6-41920722-41921457-+""","""chr6-42043482-42043608-.""",0.237817,-1.650566,126,735,3.529796


In [232]:
enhancing_or_not = multi_prom.sort("activity_difference", descending=True)\
.filter((pl.col("z_score").max().over("CRE") > 1.5) & (pl.col("z_score").min().over("CRE") < 1)
& (pl.col("z_score").min().over("CRE") > -0.5))
enhancing_or_not = enhancing_or_not.filter(pl.col("CRE").is_in(enhancing_or_not.select("CRE").unique(maintain_order=True).head(3)["CRE"]))
enhancing_or_not.head()

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
-1.086703,11,2,"""target - other""","""PLS - undefined""",267082.0,"""ANG""","""no effect""","""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",-1.379707,0.916286,283,357,4.60108
0.618555,11,1,"""negative - other""","""PLS - undefined""",64362.5,"""RNASE3""","""upregulating""","""chr14-20890827-20891165-+""","""chr14-20955217-20955500-.""",-1.287388,5.517367,283,338,4.60108
0.430336,13,1,"""negative - other""","""undefined""",166242.5,"""WFDC9""","""upregulating""","""chr20-45631225-45631710--""","""chr20-45465166-45465284-.""",-0.85562,3.611017,118,485,3.417972
0.345329,8,1,"""positive - other""","""PLS - undefined""",49139.0,"""PIGT""","""no effect""","""chr20-45415817-45416355-+""","""chr20-45465166-45465284-.""",0.289312,0.193045,118,538,3.417972
-0.483092,5,1,"""positive - other""","""PLS - ELS""",66240.5,"""PDF""","""no effect""","""chr16-69330001-69330668--""","""chr16-69396489-69396661-.""",-0.687894,0.479575,172,667,2.781948


In [230]:
silencing_or_not = multi_prom.sort("activity_difference", descending=True)\
.filter((pl.col("z_score").min().over("CRE") < -1.5) & (pl.col("z_score").max().over("CRE") > -1)
& (pl.col("z_score").max().over("CRE") < 0.5) & 
(pl.col("target_genes").n_unique().over("CRE") > 1) &
(pl.col("CRE") != "chr17-75255057-75255500-."))
silencing_or_not = silencing_or_not.filter(pl.col("CRE").is_in(silencing_or_not.select("CRE").unique(maintain_order=True).head(3)["CRE"]))
silencing_or_not

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
0.291889,8,1,"""target - other""","""PLS - ELS""",36764.0,"""RNF167""","""no effect""","""chr17-4940225-4940815-+""","""chr17-4903674-4903838-.""",0.377014,-0.277539,164,590,2.906527
-0.373544,5,1,"""target - other""","""PLS - ELS""",36205.5,"""SLC25A11""","""downregulating""","""chr17-4939699-4940224--""","""chr17-4903674-4903838-.""",0.540214,-3.184065,164,525,2.906527
-1.040084,5,2,"""target - other""","""PLS - undefined""",6898.0,"""PPIL3""","""downregulating""","""chr2-200888764-200890065--""","""chr2-200882348-200882685-.""",0.124673,-2.387872,337,1301,2.833217
0.179046,6,3,"""target - other""","""PLS - undefined""",6898.0,"""NIF3L1""","""no effect""","""chr2-200888764-200890065-+""","""chr2-200882348-200882685-.""",0.025112,0.445345,337,1301,2.833217
-1.117376,6,2,"""target - other""","""PLS - undefined""",18394.0,"""HAGH""","""downregulating""","""chr16-1827111-1827645--""","""chr16-1845606-1845938-.""",-0.049412,-2.713353,332,534,2.780738
-0.491494,23,4,"""target - other""","""PLS - undefined""",18394.0,"""FAHD1""","""no effect""","""chr16-1827111-1827645-+""","""chr16-1845606-1845938-.""",-0.512104,0.067385,332,534,2.780738


In [233]:
all_dual = pl.concat([dual_function, enhancing_or_not, silencing_or_not])
all_dual.select("CRE").unique().height

9

## Selecting enhancers and silencers

In [241]:
final_enhancers = enhancers_with_orientation.join(all_dual, on=["CRE", "promoter", "z_score"], how='anti').filter((pl.col("z_score") > 2))
final_silencers = silencers_with_orientation.join(all_dual, on=["CRE", "promoter", "z_score"], how='anti').filter((pl.col("z_score") < -2))


final_enhancers = pl.concat([
	final_enhancers.filter(pl.col("ENCODE_labels").str.contains("ELS").not_()).sort("z_score", descending=True).head(10),
	final_enhancers.filter(pl.col("ENCODE_labels").str.contains("ELS")).sort("z_score", descending=True).head(10)
])

final_silencers = pl.concat([
	final_silencers.filter(pl.col("ENCODE_labels").str.contains("ELS").not_()).sort("z_score").head(10),
	final_silencers.filter(pl.col("ENCODE_labels").str.contains("ELS")).sort("z_score").head(10)
])

## Write to files

In [243]:
final_silencers.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_silencers.tsv"), separator="\t")
final_enhancers.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_enhancers.tsv"), separator="\t")
all_dual.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_dual_function.tsv"), separator="\t")

In [ ]:
enhancers_CREs_bed = os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_enhancers_CREs.bed")
silencers_CREs_bed = os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_silencers_CREs.bed")
enhancers_promoters_bed = os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_enhancers_promoters.bed")
silencers_promoters_bed = os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_silencers_promoters.bed")

In [268]:
cd

'/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/'

In [ ]:
final_enhancers.select(pl.col("CRE_withorientation").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("CRE_withorientation") + "_" + pl.col("promoter")).write_csv(enhancers_CREs_bed,
						 separator="\t", include_header=False, quote_style='never')
final_enhancers.select(pl.col("promoter").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("CRE_withorientation") + "_" + pl.col("promoter")).write_csv(enhancers_promoters_bed,
						 separator="\t", include_header=False, quote_style='never')

final_silencers.select(pl.col("CRE_withorientation").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("CRE_withorientation") + "_" + pl.col("promoter")).write_csv(silencers_CREs_bed,
						 separator="\t", include_header=False, quote_style='never')
final_silencers.select(pl.col("promoter").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("CRE_withorientation") + "_" + pl.col("promoter")).write_csv(silencers_promoters_bed,
						 separator="\t", include_header=False, quote_style='never')

In [271]:
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_CREs.bed" > temp1 && mv temp1 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_CREs.bed"
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_CREs.bed" > temp1 && mv temp1 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_CREs.bed"
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_promoters.bed" > temp1 && mv temp1 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_promoters.bed"
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_promoters.bed" > temp1 && mv temp1 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_promoters.bed"

In [273]:
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_CREs.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers_CREs.fa"
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_CREs.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers_CREs.fa"
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_promoters.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers_promoters.fa"
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_promoters.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers_promoters.fa"